# MeSH raw data collection from DSLD — preview before saving
Collect public MeSH entry-term lookup responses, term-to-record links, and RDF statements for candidate Descriptor and Supplementary Concept Records and their owned Concepts/Terms. This is collection only: no synonym cleanup, identity acceptance, clinical claim extraction, or cross-source merging.

Use the same DSLD ingredient rows, nested rows and forms as the PubChem/RxNorm collectors. Includes co-ingredients from the ten search categories, excludes excipients. Case/whitespace query grouping reduces repeated HTTP calls while all original occurrences remain preserved.

Search exact entry-term labels. No broad substring search, fuzzy substitutions or expansion of ingredient names. An exact label hit is still an unreviewed candidate. No match means no exact entry-term result under this policy, not absence from all of MeSH. Fetch only directly owned concepts and terms; retain external hierarchy and mapping links as URIs without crawling the full ontology.

Official API documentation: https://id.nlm.nih.gov/mesh/swagger/swagger.json
RDF documentation: https://hhs.github.io/meshrdf/
Lookup service: https://www.nlm.nih.gov/pubs/techbull/ja19/brief/ja19_mesh_rdf_lookup.html

New responses remain in memory until the final save cell is enabled after reviewing the preview.


In [1]:
import json
import time
import hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from email.utils import parsedate_to_datetime
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DSLD_DIR = PROJECT_ROOT / 'data/raw/dsld'
RAW_DIR = PROJECT_ROOT / 'data/raw/mesh'
BASE_URL = 'https://id.nlm.nih.gov/mesh'
SUPPLEMENTS = ['Magnesium', 'Vitamin D', 'Ashwagandha', 'Melatonin', 'Omega-3',
               'Vitamin B12', 'Iron', 'Turmeric', 'Probiotics', 'Creatine']
MIN_INTERVAL = 0.25  # Conservative sequential request rate.
MAX_ATTEMPTS = 3
SAVE_DATA = False
session = requests.Session()
session.headers['User-Agent'] = 'SupplementRx-AI-Academic-POC/1.0'
def utcnow():
    return datetime.now(timezone.utc).isoformat()
# Reset in-memory state when rerunning configuration. Resume by rerunning later cells only.
search_results = {}
resolution_batches = []
record_batches = []
collection_started_at = utcnow()

import re
LOOKUP_LIMIT = 50
SPARQL_PAGE_SIZE = 1000
SPARQL_BATCH_SIZE = 20
MESH_URI = re.compile(r"^http://id\.nlm\.nih\.gov/mesh/[A-Z][0-9]+$")
PREFIX = "PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>\n"


## 1. Build the provenance-preserving query inventory directly from DSLD
No dependency on PubChem or RxNorm files. Retain all source names, ingredient IDs, JSON row paths and source-file hashes.


In [2]:
def walk_rows(rows, path='ingredientRows'):
    for i, row in enumerate(rows or []):
        row_path = f'{path}[{i}]'
        yield row, row_path
        yield from walk_rows(row.get('nestedRows'), row_path + '.nestedRows')

def build_inventory():
    manifest_bytes = (DSLD_DIR / 'collection_manifest.json').read_bytes()
    manifest = json.loads(manifest_bytes)
    groups = {}
    label_hashes = {}
    for label_id, categories in manifest['category_matches'].items():
        categories = sorted(set(categories) & set(SUPPLEMENTS))
        if not categories:
            continue
        path = DSLD_DIR / 'labels' / f'label_{label_id}.json'
        raw = path.read_bytes()  # Missing/invalid source records fail explicitly.
        label_hashes[label_id] = hashlib.sha256(raw).hexdigest()
        label = json.loads(raw)
        if str(label['id']) != label_id:
            raise ValueError(f'Label ID mismatch: {path}')
        for row, row_path in walk_rows(label.get('ingredientRows')):
            objects = [(row, row_path, 'ingredient')]
            objects += [(form, f'{row_path}.forms[{i}]', 'form')
                        for i, form in enumerate(row.get('forms') or [])]
            for obj, obj_path, kind in objects:
                original = obj.get('name')
                if not original or not original.strip():
                    continue
                key = ' '.join(original.casefold().split())
                group = groups.setdefault(key, {'query': original.strip(), 'variants': [],
                                               'occurrences': []})
                if original not in group['variants']:
                    group['variants'].append(original)
                group['occurrences'].append({
                    'dsld_id': label_id, 'search_categories': categories,
                    'ingredient_id': obj.get('ingredientId'), 'name': original,
                    'kind': kind, 'json_path': obj_path})
    provenance = {'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
                  'label_sha256': label_hashes}
    return dict(sorted(groups.items())), provenance

query_inventory, dsld_provenance = build_inventory()
print('Source labels:', len(dsld_provenance['label_sha256']))
print('Distinct original names:', len({v for g in query_inventory.values() for v in g['variants']}))
print('Query groups:', len(query_inventory))


Source labels: 400
Distinct original names: 753
Query groups: 739


## 2. HTTP and paginated SPARQL helpers
Preserve raw responses, request parameters, timestamps and server cache metadata. Retry transient failures with backoff. Explicitly distinguish no-match responses from HTTP and response-validation failures. Paginate SPARQL results using deterministic ordering; a failed page leaves the batch incomplete.


In [3]:
_last_request = 0.0

def request_json(path, params=None):
    global _last_request
    history = []
    url = f'{BASE_URL}/{path}'
    for attempt in range(1, MAX_ATTEMPTS + 1):
        time.sleep(max(0, MIN_INTERVAL - (time.monotonic() - _last_request)))
        _last_request = time.monotonic()
        item = {'url': url, 'params': params or {}, 'requested_at_utc': utcnow(), 'attempt': attempt}
        delay = 2 ** attempt
        retryable = True
        try:
            response = session.get(url, params=params, timeout=(10, 35))
            item.update(url=response.url, status_code=response.status_code,
                        response_headers={k:response.headers[k] for k in ('ETag','Last-Modified','Date') if k in response.headers})
            retry_after = response.headers.get('Retry-After')
            if retry_after:
                try:
                    delay = max(delay, float(retry_after))
                except ValueError:
                    try:
                        delay = max(delay, (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds())
                    except (ValueError, TypeError):
                        pass
            try:
                payload = response.json()
                item['response'] = payload
            except ValueError:
                item['response_text'] = response.text
                raise ValueError('Non-JSON response')
            if response.status_code == 200:
                if not isinstance(payload, (dict, list)) or (isinstance(payload, dict) and 'error' in payload):
                    raise ValueError('Unexpected API payload')
                history.append(item)
                return {'status': 'ok', 'response': payload, 'requests': history}
            item['error'] = f'HTTP {response.status_code}'
            retryable = response.status_code in (429, 500, 502, 503, 504)
        except (requests.RequestException, ValueError) as exc:
            item['error'] = str(exc)
        history.append(item)
        if not retryable:
            break
        if attempt < MAX_ATTEMPTS:
            time.sleep(delay)
    return {'status': 'error', 'error': history[-1]['error'], 'requests': history}


def lookup_term(term):
    result = request_json('lookup/term', {'label':term, 'match':'exact', 'limit':LOOKUP_LIMIT})
    result['term_uris'] = []
    if result['status'] != 'ok':
        return result
    rows = result['response']
    if not isinstance(rows, list) or not all(isinstance(r,dict) and isinstance(r.get('label'),str)
            and MESH_URI.fullmatch(r.get('resource','')) and r['resource'].rsplit('/',1)[-1].startswith('T') for r in rows):
        result.update(status='error', error='Invalid entry-term lookup response')
        return result
    result['term_uris'] = sorted({r['resource'] for r in rows})
    result['status'] = 'matched' if rows else 'not_found'
    result['possibly_truncated'] = len(rows) >= LOOKUP_LIMIT
    result['match_method'] = 'exact_entry_term'
    result['requires_identity_review'] = bool(rows)
    return result

def uri_values(uris):
    if not all(MESH_URI.fullmatch(uri) for uri in uris):
        raise ValueError('Unexpected MeSH resource URI')
    return ' '.join('<'+uri+'>' for uri in uris)

def query_pages(query, required):
    pages = []
    offset = 0
    while True:
        result = request_json('sparql', {'query':query, 'format':'JSON', 'inference':'false',
                                        'offset':offset, 'limit':SPARQL_PAGE_SIZE})
        pages.append(result)
        if result['status'] != 'ok':
            return {'complete':False, 'pages':pages}
        body = result['response']
        rows = body.get('results',{}).get('bindings') if isinstance(body,dict) else None
        if not isinstance(rows,list) or not all(isinstance(r,dict) and all(
                isinstance(r.get(k),dict) and 'value' in r[k] for k in required) for r in rows):
            result.update(status='error', error='Malformed SPARQL bindings')
            return {'complete':False, 'pages':pages}
        if len(rows) < SPARQL_PAGE_SIZE:
            return {'complete':True, 'pages':pages}
        offset += SPARQL_PAGE_SIZE

def bindings(batch):
    for page in batch['pages']:
        if page['status']=='ok':
            yield from page['response'].get('results',{}).get('bindings',[])


## 3. Search exact entry terms
Record current endpoint metadata. The entry-term API operates on the current dataset and does not expose a release checksum; available-year metadata does not establish a fixed snapshot. Per-request dates are retained, and queries are repeatable, but future results may differ.


In [4]:
source_metadata_start = request_json('lookup/years')
if source_metadata_start['status'] != 'ok':
    raise RuntimeError('Unable to read MeSH source metadata')
print('Available MeSH years:',source_metadata_start['response'])
for i,(key,group) in enumerate(query_inventory.items(),1):
    if search_results.get(key,{}).get('status') not in ('matched','not_found'):
        search_results[key] = lookup_term(group['query'])
    if i % 50 == 0 or i == len(query_inventory):
        print(f'Searched {i}/{len(query_inventory)}: {dict(Counter(r["status"] for r in search_results.values()))}',flush=True)
term_uris = sorted({uri for k in query_inventory for uri in search_results[k]['term_uris']})
print('Unique matched MeSH term URIs:',len(term_uris))


Available MeSH years: ['current', '2026', '2025', '2024']
Searched 50/739: {'not_found': 32, 'matched': 18}
Searched 100/739: {'not_found': 68, 'matched': 32}
Searched 150/739: {'not_found': 100, 'matched': 50}
Searched 200/739: {'not_found': 132, 'matched': 68}
Searched 250/739: {'not_found': 165, 'matched': 85}
Searched 300/739: {'not_found': 195, 'matched': 105}
Searched 350/739: {'not_found': 219, 'matched': 131}
Searched 400/739: {'not_found': 254, 'matched': 146}
Searched 450/739: {'not_found': 282, 'matched': 168}
Searched 500/739: {'not_found': 319, 'matched': 181}
Searched 550/739: {'not_found': 349, 'matched': 201}
Searched 600/739: {'not_found': 384, 'matched': 216}
Searched 650/739: {'not_found': 415, 'matched': 235}
Searched 700/739: {'not_found': 452, 'matched': 248}
Searched 739/739: {'not_found': 469, 'matched': 270}
Unique matched MeSH term URIs: 270


## 4. Collect MeSH's term-to-record relationships
Use NLM's explicit links from Terms to Concepts to Descriptor/Supplementary Concept Records. Preserve raw SPARQL bindings and all candidate links. This retrieves source relationships rather than asserting cross-database identity.


In [5]:
resolution_batches = []
for offset in range(0,len(term_uris),SPARQL_BATCH_SIZE):
    batch_terms = term_uris[offset:offset+SPARQL_BATCH_SIZE]
    query = PREFIX + 'SELECT DISTINCT ?term ?concept ?record ?type WHERE { VALUES ?term { ' + uri_values(batch_terms) + ''' }
      ?concept (meshv:term|meshv:preferredTerm) ?term .
      ?record (meshv:concept|meshv:preferredConcept) ?concept ; a ?type .
      FILTER(STRSTARTS(STR(?record), "http://id.nlm.nih.gov/mesh/D") ||
             STRSTARTS(STR(?record), "http://id.nlm.nih.gov/mesh/C"))
    } ORDER BY ?term ?record ?concept ?type'''
    batch = query_pages(query, ['term','concept','record','type'])
    batch['requested_terms'] = batch_terms
    resolution_batches.append(batch)
record_uris = sorted({r['record']['value'] for batch in resolution_batches for r in bindings(batch)})
print('Unique candidate MeSH records:',len(record_uris))
print('Incomplete resolution batches:',sum(not b['complete'] for b in resolution_batches))


Unique candidate MeSH records: 247
Incomplete resolution batches: 0


## 5. Download candidate record subgraphs
For each record collect all outgoing RDF statements on the record itself, its directly owned Concepts and their Terms. This retains scope notes, registry numbers, labels, entry terms, tree-number links, mapped headings and other available fields. No text rewriting or clinical interpretation. Store the API's raw SPARQL JSON, including literal datatype/language information.


In [6]:
record_batches = []
for offset in range(0,len(record_uris),SPARQL_BATCH_SIZE):
    batch_records = record_uris[offset:offset+SPARQL_BATCH_SIZE]
    query = PREFIX + 'SELECT DISTINCT ?record ?subject ?predicate ?object WHERE { VALUES ?record { ' + uri_values(batch_records) + ''' }
      { ?record ?predicate ?object . BIND(?record AS ?subject) }
      UNION { ?record (meshv:concept|meshv:preferredConcept) ?subject . ?subject ?predicate ?object . }
      UNION { ?record (meshv:concept|meshv:preferredConcept)/(meshv:term|meshv:preferredTerm) ?subject . ?subject ?predicate ?object . }
    } ORDER BY ?record ?subject ?predicate ?object'''
    batch = query_pages(query, ['record','subject','predicate','object'])
    batch['requested_records'] = batch_records
    record_batches.append(batch)
    print(f'Retrieved {min(offset+SPARQL_BATCH_SIZE,len(record_uris))}/{len(record_uris)} record subgraphs',flush=True)
source_metadata_end = request_json('lookup/years')


Retrieved 20/247 record subgraphs
Retrieved 40/247 record subgraphs
Retrieved 60/247 record subgraphs
Retrieved 80/247 record subgraphs
Retrieved 100/247 record subgraphs
Retrieved 120/247 record subgraphs
Retrieved 140/247 record subgraphs
Retrieved 160/247 record subgraphs
Retrieved 180/247 record subgraphs
Retrieved 200/247 record subgraphs
Retrieved 220/247 record subgraphs
Retrieved 240/247 record subgraphs
Retrieved 247/247 record subgraphs


## 6. Inspect counts, coverage, completeness and size BEFORE saving
Counts of terms, concepts, records and statements refer to distinct kinds of MeSH resources. Per-category counts overlap. No-match, unresolved, failed and potentially truncated requests are reported separately. Lookup results and RDF statements are raw candidate data awaiting later preprocessing.


In [ ]:
def make_preview():
    term_to_records = {}
    record_types = {}
    for batch in resolution_batches:
        for row in bindings(batch):
            term_to_records.setdefault(row['term']['value'],set()).add(row['record']['value'])
            record_types.setdefault(row['record']['value'],set()).add(row['type']['value'])
    query_candidates = {k:sorted({r for t in search_results[k]['term_uris'] for r in term_to_records.get(t,[])})
                        for k in query_inventory}
    categories = {}
    for category in SUPPLEMENTS:
        keys=[k for k,g in query_inventory.items() if any(category in o['search_categories'] for o in g['occurrences'])]
        categories[category]={'queries':len(keys),
            'entry_term_matches':sum(search_results[k]['status']=='matched' for k in keys),
            'queries_with_records':sum(bool(query_candidates[k]) for k in keys),
            'no_match':sum(search_results[k]['status']=='not_found' for k in keys),
            'errors':sum(search_results[k]['status']=='error' for k in keys),
            'unique_records':len({r for k in keys for r in query_candidates[k]})}
    subjects=set(); triples=set(); root_records=set(); scope_notes=set()
    for batch in record_batches:
        for row in bindings(batch):
            subjects.add(row['subject']['value'])
            triples.add(json.dumps({k:row[k] for k in ('subject','predicate','object')},sort_keys=True))
            if row['subject']['value']==row['record']['value']:
                root_records.add(row['record']['value'])
            if row['predicate']['value']=='http://id.nlm.nih.gov/mesh/vocab#scopeNote':
                scope_notes.add(row['subject']['value'])
    incomplete_records=sorted({r for b in record_batches if not b['complete'] for r in b['requested_records']} | (set(record_uris)-root_records))
    report={'source_labels':len(dsld_provenance['label_sha256']), 'query_groups':len(query_inventory),
        'search_status':dict(Counter(search_results[k]['status'] for k in query_inventory)),
        'unique_matched_entry_terms':len(term_uris), 'unique_candidate_records':len(record_uris),
        'record_types':dict(Counter(t.rsplit('#',1)[-1] for ts in record_types.values() for t in ts)),
        'queries_with_candidate_records':sum(bool(v) for v in query_candidates.values()),
        'ambiguous_queries':sum(len(v)>1 for v in query_candidates.values()),
        'possibly_truncated_queries':[k for k,r in search_results.items() if r.get('possibly_truncated')],
        'unresolved_term_uris':sorted(set(term_uris)-set(term_to_records)),
        'incomplete_resolution_batches':sum(not b['complete'] for b in resolution_batches),
        'incomplete_record_uris':incomplete_records,
        'unique_rdf_subjects':len(subjects), 'unique_rdf_statements':len(triples),
        'concepts_with_scope_notes':len(scope_notes),
        'available_years_metadata_unchanged':source_metadata_end['status']=='ok' and source_metadata_start['response']==source_metadata_end.get('response'),
        'by_category':categories}
    bundle={'source':'NLM MeSH RDF current endpoint', 'started_at_utc':collection_started_at,'finished_at_utc':utcnow(),
        'source_metadata_start':source_metadata_start,'source_metadata_end':source_metadata_end,
        'search_policy':{'match':'exact','endpoint':'lookup/term','limit':LOOKUP_LIMIT,'fuzzy':False},
        'scope':'DSLD active ingredient rows, nested rows and forms, including co-ingredients; excludes excipients',
        'dsld_provenance':dsld_provenance,'queries':query_inventory,
        'search_results':{k:search_results[k] for k in query_inventory},
        'term_resolution_responses':resolution_batches,'record_subgraph_responses':record_batches,
        'candidate_record_index':query_candidates}
    report['payload_bytes_before_report']=len(json.dumps(bundle,ensure_ascii=False).encode('utf-8'))
    bundle['report']=report
    return report,bundle

preview_report,collection_bundle=make_preview()
print(json.dumps(preview_report,indent=2))
print('Approximate payload MiB:',round(preview_report['payload_bytes_before_report']/1024**2,3))
print('No new MeSH data has been saved. Preprocessing has not been performed.')


{
  "source_labels": 400,
  "query_groups": 739,
  "search_status": {
    "not_found": 469,
    "matched": 270
  },
  "unique_matched_entry_terms": 270,
  "unique_candidate_records": 247,
  "record_types": {
    "TopicalDescriptor": 195,
    "SCR_Chemical": 49,
    "SCR_Organism": 3
  },
  "queries_with_candidate_records": 270,
  "ambiguous_queries": 6,
  "possibly_truncated_queries": [],
  "unresolved_term_uris": [],
  "incomplete_resolution_batches": 0,
  "incomplete_record_uris": [],
  "unique_rdf_subjects": 2360,
  "unique_rdf_statements": 24792,
  "concepts_with_scope_notes": 206,
  "available_years_metadata_unchanged": true,
  "by_category": {
    "Magnesium": {
      "queries": 52,
      "entry_term_matches": 29,
      "queries_with_records": 29,
      "no_match": 23,
      "errors": 0,
      "unique_records": 29
    },
    "Vitamin D": {
      "queries": 58,
      "entry_term_matches": 37,
      "queries_with_records": 37,
      "no_match": 21,
      "errors": 0,
      "unique_

## 7. Save after reviewing preview
Enable the final cell only after reviewing coverage and size. Write an immutable run with raw API responses and a checksum manifest. Completeness refers only to this exact-search collection policy, not all potentially relevant MeSH records.


In [ ]:
# SAVE_DATA = True
if SAVE_DATA:
    run_dir=RAW_DIR/'runs'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_dir.mkdir(parents=True,exist_ok=False)
    payload=json.dumps(collection_bundle,ensure_ascii=False,indent=2).encode('utf-8')
    (run_dir/'collection.json').write_bytes(payload)
    manifest={'source':'MeSH RDF','saved_at_utc':utcnow(),'collection_file':'collection.json',
        'bytes':len(payload),'sha256':hashlib.sha256(payload).hexdigest(),
        'complete':not any([preview_report['search_status'].get('error',0),preview_report['possibly_truncated_queries'],
                           preview_report['unresolved_term_uris'],preview_report['incomplete_resolution_batches'],
                           preview_report['incomplete_record_uris']]),
        'report':preview_report}
    (run_dir/'collection_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Saved:',run_dir)
else:
    print('Preview only. SAVE_DATA is False; all new responses remain in memory.')


## Executed collection result

Inspected counts, completeness and size before saving to `data/raw/mesh/runs/20261028T062833838481Z`. Saved 38,218,702 bytes (36.45 MiB) with a verified SHA-256 checksum. Searched 739 query groups from 400 DSLD labels: 270 matched entry terms and 469 no-match results. Retrieved 247 candidate records (195 TopicalDescriptor, 49 SCR_Chemical, 3 SCR_Organism), containing 24,792 distinct RDF statements across 2,360 resources; 206 concepts have scope notes. Six queries link to multiple candidate records. There were no request errors, unresolved terms, incomplete record batches or lookup-limit warnings.

Preserved raw lookup responses and RDF bindings, including source text, datatypes, language tags and relationship URIs. No preprocessing or cross-source merging was performed. The API uses current MeSH data; available-year metadata remained unchanged, but is not a fixed-release checksum. `SAVE_DATA` remains `False` for future runs.
